In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

# 0. Cargar Datos

In [31]:
# Cargar datos
columnas_segmentacion = ["Año", "Mes", "Cod Cliente", "Cod SKU", "Nombre Consolidado", "Zona"]
path_ventas = "datos/Ventas por Cliente/ventas_con_precio_lista_y_descuentos_2025_2026.csv"
ventas = pd.read_csv(path_ventas)#, usecols=columnas_segmentacion)
print("Ventas - Filas:", ventas.shape[0])
ventas.head()

/var/folders/dw/9r647bwd1kl_g5lxtz0kh4100000gp/T/ipykernel_68056/2410920425.py:4: DtypeWarning: Columns (0: dscto_volumen, 1: ids_descuento_volumen) have mixed types. Specify dtype option on import or set low_memory=False.
  ventas = pd.read_csv(path_ventas)#, usecols=columnas_segmentacion)


Ventas - Filas: 10596574


,Año,Mes,Cod Canal Comercial,Clase Factura,Fecha Factura,Cod Cliente,Nombre Cliente Padre,Cod Consolidado,Nombre Consolidado,Nombre Familia,...,Distrito,Precio_Lista,dscto_base,id_descuento_base,dscto_volumen,ids_descuento_volumen,dscto_binario,id_descuento_binario,carta_impacto,id_descuento_carta_impacto
0,2025,1,CB,ZV01,2025-01-02,1145633,NaN,32,COBERTURA,SALAMES,...,PUNTA ARENAS,11538.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,HR,ZV01,2025-01-03,43790,NaN,55,HORECA VOLUMEN,MORTADELAS,...,SANTIAGO HORECA PAP,2715.0,-3.0,2352.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,HR,ZV01,2025-01-03,1227049,NaN,37,OTROS HORECA,VIENESAS,...,SANTIAGO HORECA PAP,2722.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,1,CB,ZV01,2025-01-02,1174760,NaN,32,COBERTURA,HAMBURGUESAS,...,NaN,5171.0,-12.5,2372.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,HR,ZV01,2025-01-03,1015869,NaN,55,HORECA VOLUMEN,PARRILLEROS,...,SANTIAGO HORECA PAP,3232.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
ventas.columns

Index(['Año', 'Mes', 'Cod Canal Comercial', 'Clase Factura', 'Fecha Factura',
       'Cod Cliente', 'Nombre Cliente Padre', 'Cod Consolidado',
       'Nombre Consolidado', 'Nombre Familia', 'N° Factura', 'Nombre Marca',
       'Nombre Tipo Carne', 'Cod SKU', 'Nombre SKU', 'Factura Venta',
       'Factura Kilos', 'Kilo Real', 'Kilos Nc', 'Monto Nc', 'Monto Real',
       'Precio', 'venta_id', 'Zona', 'Distrito', 'Precio_Lista', 'dscto_base',
       'id_descuento_base', 'dscto_volumen', 'ids_descuento_volumen',
       'dscto_binario', 'id_descuento_binario', 'carta_impacto',
       'id_descuento_carta_impacto'],
      dtype='str')

In [33]:
# Filtrar canales relevantes
canales_relevantes = [
    "COBERTURA",
    "VOLUMEN COBERTURA",
    # "MAYORISTAS CADENAS",
    # "MAYORISTA B VOLUMEN",
    # "OTROS MAYORISTAS",
    # "HORECA VOLUMEN",
    # "OTROS HORECA",
]
ventas = ventas[
    (ventas["Cod Canal Comercial"] == "CB") &
    (ventas["Nombre Consolidado"].isin(canales_relevantes))
    ]
print("Ventas después de filtrar canales irrelevantes - Filas:", ventas.shape[0])
ventas.head()

Ventas después de filtrar canales irrelevantes - Filas: 9615996


,Año,Mes,Cod Canal Comercial,Clase Factura,Fecha Factura,Cod Cliente,Nombre Cliente Padre,Cod Consolidado,Nombre Consolidado,Nombre Familia,...,Distrito,Precio_Lista,dscto_base,id_descuento_base,dscto_volumen,ids_descuento_volumen,dscto_binario,id_descuento_binario,carta_impacto,id_descuento_carta_impacto
0,2025,1,CB,ZV01,2025-01-02,1145633,NaN,32,COBERTURA,SALAMES,...,PUNTA ARENAS,11538.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,1,CB,ZV01,2025-01-02,1174760,NaN,32,COBERTURA,HAMBURGUESAS,...,NaN,5171.0,-12.5,2372.0,NaN,NaN,NaN,NaN,NaN,NaN
7,2025,1,CB,ZV01,2025-01-02,1137411,NaN,54,VOLUMEN COBERTURA,HORTALIZAS,...,SANTIAGO CENTRO COSTA,2198.0,-3.0,2837.0,NaN,NaN,NaN,NaN,NaN,NaN
9,2025,1,CB,ZV01,2025-01-02,1134837,NaN,32,COBERTURA,PATE,...,SANTIAGO CENTRO COSTA,4654.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,2025,1,CB,ZV01,2025-01-02,1114635,NaN,54,VOLUMEN COBERTURA,PATE,...,SANTIAGO CENTRO COSTA,4654.0,-3.0,1277.0,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
path_info_zonales = "datos/Clientes - Zonales.xlsx"
zonales = pd.read_excel(path_info_zonales)
zonales.head()

,Id cliente,REGIONES
0,1194043,RANCAGUA
1,1155918,RANCAGUA
2,1090050,RANCAGUA
3,1203751,RANCAGUA
4,1231602,IQUIQUE


In [35]:
zonales.columns

Index(['Id cliente', 'REGIONES'], dtype='str')

In [36]:
path_clientes = "datos/Base Datos Clientes - Carga de Trabajo.xlsx"
clientes = pd.read_excel(path_clientes, header=1)
clientes.head()

,Zona,Distrito,CodJV,CodSup,CodVend,NombreVendedor,Canal,SubCanal,CodCliente Padre,Cliente Padre,...,MA,MI,JU,VI,SA,Frecuencia,Tratamiento,CodPago,DesPago,Estado
0,SANTIAGO,SANTIAGO CENTRO COSTA,854,854,1084869,ALEXIS PEREZ MUÑOZ,COBERTURA,COBERTURA,0,-,...,0.0,0.0,0.0,1.0,0,1.0,Cliente,C005,FIRMA 7 DIAS,1
1,SUR 1,RANCAGUA,481,481,482,JUAN FARIAS PEREZ,COBERTURA,COBERTURA,0,-,...,0.0,0.0,0.0,0.0,0,0.5,Cliente,C000,EFECTIVO - CHEQUE AL DIA,1
2,SUR 2,CONCEPCION SUR,1178566,1178566,1084857,ALEX RIOS CUEVAS,COBERTURA,COBERTURA,0,-,...,0.0,0.0,0.5,0.0,0,0.5,Cliente,C017,EFECTIVO,1
3,SUR 1,TALCA,581,581,1160963,RODRIGO AVENDAÑO HERNANDEZ,COBERTURA,COBERTURA,0,-,...,0.0,1.0,0.0,0.0,0,1.0,Cliente,C044,PAGO ELECT CREDITO 07 DIAS,1
4,NORTE 2,VIÑA COSTA,1216438,1216438,1206012,PAULO RUZ BILBAO,COBERTURA,COBERTURA,0,-,...,1.0,0.0,0.0,1.0,0,2.0,Cliente,C044,PAGO ELECT CREDITO 07 DIAS,1


In [37]:
clientes.columns

Index(['Zona', 'Distrito', 'CodJV', 'CodSup', 'CodVend', 'NombreVendedor',
       'Canal', 'SubCanal', 'CodCliente Padre', 'Cliente Padre', 'CodCliente',
       'RazonSocial', 'Direccion', 'Comuna', 'TipoNeg', 'DesTipoNeg', 'Relev',
       'NivPrecio', 'Telefono', 'Correo', 'VtaUlt3M', 'PromUlt3M',
       'Margen3ULTM', 'PromMargen3ULTM', 'CodRuta', 'DesRuta', 'NroSec',
       'RitmoVisita', 'LU', 'MA', 'MI', 'JU', 'VI', 'SA', 'Frecuencia',
       'Tratamiento', 'CodPago', 'DesPago', 'Estado'],
      dtype='str')

# 1. Cruces de bases y filtros

In [38]:
ventas = ventas.merge(zonales[["Id cliente", "REGIONES"]].rename(columns={"Id cliente": "Cod Cliente", "REGIONES": "zonal"}), on="Cod Cliente", how="left")

In [39]:
# Filtrar a solo segundo semestre de 2025
ventas = ventas[ventas["Año"] == 2025]
ventas = ventas[ventas["Mes"] >= 7]
print("Ventas después de filtrar fechas - Filas:", ventas.shape[0])
ventas.head()

Ventas después de filtrar fechas - Filas: 3813911


,Año,Mes,Cod Canal Comercial,Clase Factura,Fecha Factura,Cod Cliente,Nombre Cliente Padre,Cod Consolidado,Nombre Consolidado,Nombre Familia,...,Precio_Lista,dscto_base,id_descuento_base,dscto_volumen,ids_descuento_volumen,dscto_binario,id_descuento_binario,carta_impacto,id_descuento_carta_impacto,zonal
37,2025,8,CB,ZV01,2025-08-18,1037715,NaN,54,VOLUMEN COBERTURA,TIPICOS,...,5171.0,-3.0,2086.0,NaN,NaN,NaN,NaN,NaN,NaN,ANTOFAGASTA
41,2025,8,CB,ZV01,2025-08-15,1217828,NaN,32,COBERTURA,TIPICOS,...,5171.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TEMUCO
50,2025,10,CB,ZV01,2025-10-06,1230617,NaN,32,COBERTURA,QUESOS,...,5171.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VIÑA DEL MAR
51,2025,10,CB,ZV01,2025-10-06,239801,NaN,54,VOLUMEN COBERTURA,QUESOS,...,5171.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
57,2025,12,CB,ZV01,2025-12-27,1201468,NaN,32,COBERTURA,MORTADELAS,...,3717.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TEMUCO


In [40]:
# Excluir clientes por TipoNeg antes de armar la matriz cliente-SKU
tipos_neg_excluidos = ["MO", "SN", "SR", "CD", "CZ", "DG", "EC", "PC", "HR"]
path_clientes = "datos/Base Datos Clientes - Carga de Trabajo.xlsx"
clientes_tiponeg = pd.read_excel(path_clientes, header=1, usecols=["CodCliente", "TipoNeg"])

ventas = ventas.merge(
    clientes_tiponeg.drop_duplicates(subset=["CodCliente"]),
    left_on="Cod Cliente",
    right_on="CodCliente",
    how="left",
)
ventas = ventas.loc[~ventas["TipoNeg"].isin(tipos_neg_excluidos)].drop(columns=["CodCliente", "TipoNeg"])
print("Ventas después de filtrar TipoNeg excluidos - Filas:", ventas.shape[0])
ventas.head()

Ventas después de filtrar TipoNeg excluidos - Filas: 3813911


,Año,Mes,Cod Canal Comercial,Clase Factura,Fecha Factura,Cod Cliente,Nombre Cliente Padre,Cod Consolidado,Nombre Consolidado,Nombre Familia,...,Precio_Lista,dscto_base,id_descuento_base,dscto_volumen,ids_descuento_volumen,dscto_binario,id_descuento_binario,carta_impacto,id_descuento_carta_impacto,zonal
0,2025,8,CB,ZV01,2025-08-18,1037715,NaN,54,VOLUMEN COBERTURA,TIPICOS,...,5171.0,-3.0,2086.0,NaN,NaN,NaN,NaN,NaN,NaN,ANTOFAGASTA
1,2025,8,CB,ZV01,2025-08-15,1217828,NaN,32,COBERTURA,TIPICOS,...,5171.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TEMUCO
2,2025,10,CB,ZV01,2025-10-06,1230617,NaN,32,COBERTURA,QUESOS,...,5171.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VIÑA DEL MAR
3,2025,10,CB,ZV01,2025-10-06,239801,NaN,54,VOLUMEN COBERTURA,QUESOS,...,5171.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
4,2025,12,CB,ZV01,2025-12-27,1201468,NaN,32,COBERTURA,MORTADELAS,...,3717.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TEMUCO


In [41]:
ventas["Cod Canal Comercial"].value_counts()

Cod Canal Comercial
CB    3813911
Name: count, dtype: int64

# 2. Crear segmentación

In [42]:
parametros_kmeans = {
    "random_state": 42,
    "batch_size": 4096,
    "n_init": "auto",
}

In [43]:
def segmentar(
        ventas_df,
        granularidad_sku,   # Familia, Familia-marca, Subfamilia, SKU
        conteo_compra,      # Binario, porcentaje de ingresos
        ks_clustering,      # Set de k's a probar con kmeans
        verbosity=False     # Mostrar proceso o no
):
    # Armar matriz cliente-SKU una sola vez por combinación cliente-producto
    if not isinstance(ventas_df, pd.DataFrame):
        raise TypeError("ventas_df debe ser un DataFrame de pandas. Reejecuta las celdas de preparación de ventas.")
    columna_producto = granularidad_sku
    cols = ["Cod Cliente", granularidad_sku]
    columnas_faltantes = [col for col in cols if col not in ventas_df.columns]
    if columnas_faltantes:
        raise KeyError(f"Columnas faltantes en ventas_df: {columnas_faltantes}")
    compras_cliente_sku = ventas_df[cols]
    if conteo_compra == "binario":
        matriz_cliente_sku = pd.crosstab(
            compras_cliente_sku["Cod Cliente"],
            compras_cliente_sku[columna_producto],
        ).astype(np.uint8)
    # TODO: nivel_compra como porcentaje de los ingresos
    elif conteo_compra == "pg_ingresos":
        return
    else:
        raise ValueError(f"conteo_compra no soportado: {conteo_compra}")
    if verbosity:
        print("Matriz cliente-SKU:", matriz_cliente_sku.shape)

    # Normalizar aproxima clustering por similitud coseno sin construir una matriz cliente-cliente
    X_clientes = csr_matrix(matriz_cliente_sku.to_numpy(dtype=np.float32))
    X_clientes = normalize(X_clientes, norm="l2", copy=False)
    if verbosity:
        print("Matriz sparse normalizada:", X_clientes.shape, "nnz=", X_clientes.nnz)

    k_max = X_clientes.shape[0] - 1
    rango_clusters = [k for k in ks_clustering if 2 <= k <= k_max]
    if not rango_clusters:
        raise ValueError(
            f"No hay valores validos en ks_clustering para {X_clientes.shape[0]} clientes. Usa k entre 2 y {k_max}."
        )

    # Usamos una muestra para calcular score para reducir tiempo de computo
    tamano_muestra_silhouette = min(30000, X_clientes.shape[0])
    if verbosity:
        print(f"Calculando silhouette sobre una muestra de {tamano_muestra_silhouette} clientes")

    # Iteramos sobre los K's para llegar a la mejor clusterización
    resultados_silhouette = []
    for k in rango_clusters:
        modelo_tmp = MiniBatchKMeans(
            n_clusters=k,
            **parametros_kmeans,
        )
        labels_tmp = modelo_tmp.fit_predict(X_clientes)
        score_tmp = silhouette_score(
            X_clientes,
            labels_tmp,
            metric="cosine",
            sample_size=tamano_muestra_silhouette,
            random_state=42,
        )
        resultados_silhouette.append({
            "k": k,
            "silhouette_score": score_tmp,
        })
        if verbosity:
            print(f"k={k}: silhouette={score_tmp:.4f}")

    resultados_silhouette = pd.DataFrame(resultados_silhouette)
    num_clusters = int(
        resultados_silhouette.loc[
            resultados_silhouette["silhouette_score"].idxmax(),
            "k",
        ]
    )
    if verbosity:
        print(f"Mejor número de clusters según silhouette: {num_clusters}")
        plt.figure(figsize=(8, 4))
        plt.plot(
            resultados_silhouette["k"],
            resultados_silhouette["silhouette_score"],
            marker="o",
        )
        plt.axvline(num_clusters, color="red", linestyle="--", label=f"Mejor k = {num_clusters}")
        plt.xlabel("Número de clusters (k)")
        plt.ylabel("Silhouette score")
        plt.title("Búsqueda de k óptimo para segmentación de clientes")
        plt.legend()
        plt.show()

    modelo_clusters = MiniBatchKMeans(
        n_clusters=num_clusters,
        **parametros_kmeans,
    )
    clusters = modelo_clusters.fit_predict(X_clientes) + 1

    segmentacion_clientes = pd.DataFrame({
        "Cod Cliente": matriz_cliente_sku.index,
        "cluster": clusters,
    })
    return matriz_cliente_sku, segmentacion_clientes, resultados_silhouette

In [44]:
# Iteramos la clusterización
granularidad_sku = "Cod SKU"             # Familia, Familia-marca, Subfamilia, SKU
conteo_compra = "binario"                # Binario, porcentaje de ingresos
ks_clustering = range(2, 15)
verbosity = False
matriz_cliente_sku, segmentacion_clientes, resultados_silhouette = segmentar(
    ventas,
    granularidad_sku,
    conteo_compra,
    ks_clustering,
    verbosity
)

In [45]:
# Vemos la distribución de zonas entre clusters

# 2. Analisis de Clusters

In [46]:
path_segmentacion_prods = "datos/segmentacion_productos.xlsx"
segmentacion_productos = pd.read_excel(path_segmentacion_prods)

In [47]:
def obtener_canastas(
        cluster_elegido,
        matriz_cliente_sku,
        segmentacion_clientes,
        granularidad_sku,    # Familia, Familia-marca, Subfamilia, SKU
        umbral=0.5,          # A partir de qué porcentaje de pertenencia considero un producto como miembro de la canasta
        verbosity=False
):
    clientes_cluster = segmentacion_clientes[segmentacion_clientes["cluster"] == cluster_elegido]["Cod Cliente"]
    skus_cluster = matriz_cliente_sku.loc[clientes_cluster]

    tabla_skus_comunes_cluster = (
        skus_cluster.sum()
        .sort_values(ascending=False)
        .rename_axis(granularidad_sku)
        .reset_index(name="conteo_clientes")
    )
    cantidad_clientes_cluster = len(clientes_cluster)
    tabla_skus_comunes_cluster["porcentaje_clientes_cluster"] = (
        tabla_skus_comunes_cluster["conteo_clientes"] / cantidad_clientes_cluster
    )
    tabla_skus_comunes_cluster = tabla_skus_comunes_cluster[
        [granularidad_sku, "nombre_sku", "conteo_clientes", "porcentaje_clientes_cluster"]
    ]

    if verbosity:
        display(tabla_skus_comunes_cluster.head(n=20))

    skus_canasta = tabla_skus_comunes_cluster[
        tabla_skus_comunes_cluster["porcentaje_clientes_cluster"] > umbral
    ][granularidad_sku].unique()

    if verbosity:
        print(f"Canasta del cluster: {skus_canasta}")

    return skus_canasta

In [48]:
# Para cada cluster armar su canasta

In [49]:
# Armar matriz de clusters_clusters calculando Jaccard de las canastas

# 4. Análisis de Clusters resultantes

In [50]:
# AgregarTipoNeg a segmentacion_clientes
segmentacion_clientes = segmentacion_clientes.merge(clientes[["CodCliente", "DesTipoNeg"]], left_on="Cod Cliente", right_on="CodCliente", how="left")
segmentacion_clientes.drop(columns=["CodCliente"], inplace=True)
segmentacion_clientes.head()

,Cod Cliente,cluster,DesTipoNeg
0,235,6,AL - ALMACÉN
1,499,6,AL - ALMACÉN
2,547,3,AL - ALMACÉN
3,665,2,AL - ALMACÉN
4,1012,4,AL - ALMACÉN


In [51]:
# Tabla con porcentaje de clientes por TipoNeg en cada cluster
segmentacion_tiponeg = segmentacion_clientes.copy()
segmentacion_tiponeg["DesTipoNeg"] = segmentacion_tiponeg["DesTipoNeg"].fillna("Sin TipoNeg")

clusters_ordenados = sorted(segmentacion_tiponeg["cluster"].dropna().unique())

tabla_tiponeg_cluster = (
    pd.crosstab(
        segmentacion_tiponeg["DesTipoNeg"],
        segmentacion_tiponeg["cluster"],
        normalize="index",
    )
    .reindex(columns=clusters_ordenados, fill_value=0)
    .mul(100)
    .round(1)
)

tabla_tiponeg_cluster.columns = [f"Cluster {int(col)}" for col in tabla_tiponeg_cluster.columns]

cluster_predominante = tabla_tiponeg_cluster.idxmax(axis=1)
porcentaje_predominante = tabla_tiponeg_cluster.max(axis=1)

tabla_tiponeg_cluster["Cluster predominante (%)"] = (
    cluster_predominante
    + " ("
    + porcentaje_predominante.round(1).astype(str)
    + "%)"
)

tabla_tiponeg_cluster

,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9,Cluster 10,Cluster predominante (%)
DesTipoNeg,,,,,,,,,,,
AL - ALMACÉN,7.0,9.7,11.4,8.5,13.5,13.6,7.5,6.8,8.7,13.4,Cluster 6 (13.6%)
CA - CARNICERÍA,9.0,5.4,5.5,6.9,40.3,9.1,3.0,4.7,2.5,13.7,Cluster 5 (40.3%)
FI - FIAMBRERÍA,18.1,11.2,4.4,5.6,18.8,16.2,1.9,16.9,3.8,3.1,Cluster 5 (18.8%)
FS - FUENTEDE SODA,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Cluster 1 (100.0%)
MA - MAYORISTA,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Cluster 1 (100.0%)
MK - MINIMARKET,5.4,22.8,3.4,7.8,27.7,8.9,3.8,11.4,3.2,5.6,Cluster 5 (27.7%)
PA - PANADERÍA,6.8,16.8,7.0,6.7,7.3,16.7,8.2,18.9,9.4,2.1,Cluster 8 (18.9%)
PZ - PIZZERIA,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,Cluster 5 (100.0%)
SL - SUPERMERCADO LOCAL,2.4,33.3,0.5,1.4,43.0,10.1,1.0,7.2,0.5,0.5,Cluster 5 (43.0%)
